**Author**: Felipe Matheus
**Pipeline**: Uncertainty-aware MBC — **Cold drawing route selection**

Probabilistic counterpart of the deterministic `mbc_cold-drawing_uts` pipeline.
Same combinatorial skeleton (mathematical_formulation_cd_DETERMINISTIC.pdf),
new predictive core:

1. **Graph**: `CDGraph.generate_all_sequences_df` enumerates every admissible
   route D0 → Df (steps ∈ S, reduction ratio ≤ rr_max, ≤ max_passes) — the
   SAME ported DAG/DFS code as the deterministic MBC.
2. **Score**: each route is rolled out with **K Monte Carlo trajectories**
   (`ColdDrawingRollout.predict_sequences`): every pass samples the calibrated
   truncated predictive law and feeds the next (spec §4.3). Multi-surrogate
   ready — state features and same-pass upstreams (grain size → IACS/UTS) are
   detected STRUCTURALLY from the bundles, nothing hardcoded.
3. **Feasibility**: chance constraint on the FINAL state. Because the K final
   samples share trajectories across targets, `pr_success_all` is a **joint**
   empirical probability (no independence assumption — an upgrade over the
   annealing grid's product rule).
4. **Ranking**: highest `pr_success_all`, then fewest passes, then smallest
   cumulative reduction — the deterministic criteria, risk-aware.
5. **Export**: `pev.export_routes` — shared with
   `script_mbc_cold_drawing_uncertainty.py` (fix once, fixed everywhere).


# 1. Setup

In [ ]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.processing.Processing import Processing
from src.feature_engineering.FeatureEngineering import FeatureEngineering
from src.feature_engineering.CDHelper import CDHelper
from src.modeling.Modeling import Modeling
from src.modeling.MBCInference import MBCInference
from src.modeling.CDRollout import ColdDrawingRollout
from src.modeling.CDGraph import ColdDrawingMBCHelper
from src.metrics.Evaluation import Evaluation
from src.metrics.ProbabilisticEvaluation import ProbabilisticEvaluation
from src.DataLoader import LoaderHelper
from config.Variables import Variables

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s")

%load_ext autoreload
%autoreload 2

varv = Variables()
proc = Processing()
feng = FeatureEngineering()
cdh  = CDHelper()
modl = Modeling()
evla = Evaluation()
load = LoaderHelper()
mbc  = MBCInference(modl, load)
roll = ColdDrawingRollout(modl)
pev  = ProbabilisticEvaluation()


# 2. Configuration

In [ ]:
# ---- Surrogate bundles (TAG dirs; best run auto-picked by RMSE) ----
# Add "iacs"/"grain_size" entries when those cold-drawing bundles exist; the
# rollout detects state features and same-pass chaining by itself.
BUNDLE_TAG_DIRS = {
    "uts": os.path.join(varv.PATHS.models, "cold_drawing_uts",
                        "experiments", "cd-uts-v1-best_quality"),
    # "grain_size": os.path.join(varv.PATHS.models, "cold_drawing_grain_size",
    #                            "experiments", "cd-gs-v1-best_quality"),
    # "iacs": os.path.join(varv.PATHS.models, "cold_drawing_iacs",
    #                      "experiments", "cd-iacs-v1-best_quality"),
}

# ---- Fixed material state (every non-geometry, non-chained feature) ----
# original_* doubles as the pass-1 fallback for the chained state features
# (initial_tensile_strength <- original_tensile_strength, etc.).
MATERIAL_PROPERTIES = {
    "purity": 99.9,
    "original_tensile_strength": 280.0,
    # "original_grain_size": 40.0,
}
INIT_STATE = {}          # explicit pass-1 state overrides, e.g. {"grain_size": 40.0}

# ---- Route generation (mirrors the deterministic YAML) ----
ORIGINAL_DIAMETER = 2.0      # mm
FINAL_DIAMETER    = 1.2      # mm
STEPS             = [0.1, 0.2, 0.3, 0.4]
RR_MAX            = 80.0     # % per pass
MAX_PASSES        = 20
MAX_SEQUENCES     = 2000

# ---- Probabilistic acceptance on the FINAL state ----
MIN_SETPOINTS = {"tensile_strength": 445.0}
MAX_SETPOINTS = {}           # e.g. {"grain_size": 30.0} when the GS bundle exists
DELTA = 0.10                 # keep routes with Pr(all) >= 1 - delta

# ---- Monte Carlo ----
K_SAMPLES = 200
SEED = 0

# ---- Export ----
IDENTIFIER = "cd_routes_pilot"
TOP_N_EXPORT = 20


# 3. Load surrogates + validate setpoints (fail fast)

In [ ]:
bundles = mbc.load_surrogates_best(BUNDLE_TAG_DIRS, metric="rmse")
targets = pev.validate_setpoints(bundles, MIN_SETPOINTS,
                                 max_setpoints=MAX_SETPOINTS or None)
for key, b in bundles.items():
    print(f"[{key}] target={b.target} | features={b.features}")


## 3.1 Detect the chained state features

Structural, zero hardcode: a feature named `<short>` or `initial_<short>` is
previous-pass state fed by that surrogate; a feature named `<short>_final`
in ANOTHER bundle is a same-pass upstream feed (fixes within-pass order).

In [ ]:
prev_map, same_map, order = roll.detect_state_features(bundles)
print("previous-pass state:", prev_map)
print("same-pass upstream :", same_map)
print("within-pass order  :", order)


# 4. Enumerate admissible routes (ported deterministic graph)

In [ ]:
sequences_df = ColdDrawingMBCHelper.generate_all_sequences_df(
    original_diameter=ORIGINAL_DIAMETER,
    final_diameter=FINAL_DIAMETER,
    steps=STEPS,
    rr_max=RR_MAX,
    max_passes=MAX_PASSES,
    max_sequences=MAX_SEQUENCES,
)
n_routes = sequences_df.index.get_level_values("sequence_id").nunique()
print(f"{n_routes} admissible routes ({ORIGINAL_DIAMETER} -> {FINAL_DIAMETER} mm)")
sequences_df.head(8)


# 5. Score every route — K Monte Carlo trajectories per route

Cost: `n_routes x n_passes x n_surrogates` batched predictor calls of K rows.
`finals` keeps the K SHARED final-state samples per route for the joint
feasibility below.

In [ ]:
df_routes, finals = roll.predict_sequences(
    bundles, sequences_df,
    fixed_state=MATERIAL_PROPERTIES,
    init_state=INIT_STATE or None,
    k_samples=K_SAMPLES, seed=SEED,
)
df_routes.head()


# 6. Chance-constrained feasibility on the final state

In [ ]:
df_prob = pev.add_route_success_probabilities(
    df_routes, finals, MIN_SETPOINTS, bundles,
    max_setpoints=MAX_SETPOINTS or None,
)
df_feasible = pev.apply_probabilistic_setpoints(df_prob, delta=DELTA)
print(f"Feasible routes (Pr >= {1-DELTA:.2f}): {len(df_feasible)} / {len(df_prob)}")
df_prob[["route", "n_passes", "cum_reduction"]
        + [f"{t}_final_mu" for t in targets]
        + [f"pr_success_{t}" for t in targets]
        + ["pr_success_all"]].sort_values("pr_success_all", ascending=False).head(10)


# 7. Rank the survivors

Highest joint Pr(success) first, then process simplicity (fewer passes,
smaller cumulative die reduction) — the deterministic criteria, risk-aware.

In [ ]:
ranked = pev.rank_routes(df_feasible if len(df_feasible) else df_prob)
ranked.head(10)


# 8. Inspect the winning route pass by pass

Re-roll the top route and plot the predicted evolution with the honest
compounded band μ ± σ — the plot a stakeholder reads.

In [ ]:
top_sid = int(ranked.iloc[0]["sequence_id"])
route_top = sequences_df.loc[top_sid].reset_index()
per_pass, _ = roll.rollout_route_mc(
    bundles, route_top,
    fixed_state=MATERIAL_PROPERTIES, init_state=INIT_STATE or None,
    k_samples=K_SAMPLES, seed=SEED,
)
print("Route:", ranked.iloc[0]["route"])

fig, axes = plt.subplots(1, len(targets), figsize=(6.2 * len(targets), 4.2),
                         squeeze=False)
for ax, t in zip(axes[0], targets):
    mu = per_pass[f"{t}_mu"]; sg = per_pass[f"{t}_sigma"]
    x = per_pass["pass_number"]
    ax.plot(x, mu, "o-", label=f"{t} μ")
    ax.fill_between(x, mu - sg, mu + sg, alpha=.25, label="±1σ (compounded)")
    if t in MIN_SETPOINTS:
        ax.axhline(MIN_SETPOINTS[t], color="r", ls="--", lw=1,
                   label=f"min setpoint {MIN_SETPOINTS[t]:g}")
    if t in (MAX_SETPOINTS or {}):
        ax.axhline(MAX_SETPOINTS[t], color="darkred", ls=":", lw=1,
                   label=f"max setpoint {MAX_SETPOINTS[t]:g}")
    ax.set_xlabel("pass"); ax.set_ylabel(t); ax.legend(); ax.grid(alpha=.3)
plt.suptitle("Top route — predicted evolution with compounded uncertainty")
plt.tight_layout(); plt.show()


# 9. Export (shared method — same call as the standalone script)

In [ ]:
output_dir = Path(varv.PATHS.data_enriched) / "mbc_cold_drawing_uncertainty"
out_path = pev.export_routes(
    ranked, output_dir, IDENTIFIER, targets=targets, top_n=TOP_N_EXPORT,
)
print(f"Exported to: {out_path}")

# full audit dump (every scored route with probabilities)
full_path = output_dir / IDENTIFIER / "mbc_cd_unc_all_routes.csv"
pev.rank_routes(df_prob).to_csv(full_path, index=False)
print(f"Full audit grid: {full_path}")
